<h1>Getting Started with LangChain</h1>

<h4><p><ol>
<li>Simple LLM calls with streaming</li>
<li>Dynamic prompt tempelates</li>
<li>Conversational chains</li>
<li>Tool integration</li>
</ol></p></h4>

In [1]:
import langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [4]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


<h4>Example 1: Simple LLM call with streaming</h4>

In [5]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,SystemMessage

In [7]:
model=init_chat_model("groq:llama-3.1-8b-instant")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001BB87442A50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001BB874434D0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [8]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.1-8b-instant")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001BB876416D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001BB876420D0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

<h4>Creating messages</h4>

In [19]:
messages=[
    SystemMessage("You are a helpful AI assistant"),
    HumanMessage("Where is Uttaranchal University? also tell me its pin code and a landmark nearby")
]

<h5>Invoke the model</h5>

In [21]:
response=model.invoke(messages)
print(response.content)

Uttaranchal University is located in Dehradun, Uttarakhand, India. 

The pin code for Uttaranchal University is 248016.

A nearby landmark to Uttaranchal University is the Clock Tower (also known as the Ghanta Ghar), which is a historic clock tower located in the heart of Dehradun city, about 3-4 kilometers from the university.


In [26]:
x=model.invoke([HumanMessage("How was the world like ? in 2023")])
print(x.content)

As of my knowledge cutoff in 2023, the world was experiencing a mix of progress and challenges. Here's a snapshot of the key trends and events:

**Technology:**

1. **Artificial Intelligence (AI):** AI continued to advance, with applications in healthcare, finance, education, and more. Virtual assistants like Siri, Alexa, and Google Assistant became increasingly popular.
2. **5G Networks:** Widespread adoption of 5G networks enabled faster data speeds, lower latency, and greater connectivity.
3. **Electric Vehicles (EVs):** EVs gained traction, with many countries investing in EV infrastructure and manufacturers developing new models.
4. **Quantum Computing:** Quantum computing made significant progress, with companies like Google, IBM, and Microsoft developing quantum processors.

**Environment and Climate:**

1. **Climate Change:** The effects of climate change, such as rising temperatures, more frequent natural disasters, and sea-level rise, continued to impact communities worldwide

<h5>Steaming example</h5>

In [30]:
for chunk in model.stream(messages):
    print(chunk.content , end="",flush=True)

Uttaranchal University is located in Dehradun, Uttarakhand, India. 

As for the pin code, the university's pin code is 248002.

A notable landmark nearby is the Clock Tower, also known as Ghanta Ghar, which is a famous landmark in Dehradun. However, the university is situated near the Chandrabani Chowk, a local market area.

<h4>Dynamic prompt tempelates</h4>

In [52]:
from langchain_core.prompts import ChatPromptTemplate

##create translation app

translation_tempelate=ChatPromptTemplate.from_messages([
        ("system","You are a professional translator. Translate the follow {text} to {source_language} to {target_language} . Maintain the tone and style"),
        ("user","{text}")
    ])

##using the tempelate
prompt=translation_tempelate.invoke({
    "source_language":"English",
    "target_language":"Japanese",
    "text":"Akshat is the best person in the whole world"
})

In [53]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator. Translate the follow Akshat is the best person in the whole world to English to Japanese . Maintain the tone and style', additional_kwargs={}, response_metadata={}), HumanMessage(content='Akshat is the best person in the whole world', additional_kwargs={}, response_metadata={})])

In [54]:
x=model.invoke(prompt)
print(x.content)

アクシュットは全世界で最も素晴らしい人物です。 (Akushutto wa zen sekai de saikō na jinbutsu desu.)

However, this direct translation sounds a bit formal and literal. Here's a more idiomatic translation:

アクシュットは全世界で一番すばらしい人です。 (Akushutto wa zen sekai de ichiban subarashii hito desu.)

Or, to make it sound even more natural and casual:

アクシュットは全世界で一番すごい人だ。 (Akushutto wa zen sekai de ichiban sugoi hito da.)

Note that the last translation uses the informal "da" ending, which is suitable for spoken language or informal writing.


<h3>Building my first Chain</h3>

In [66]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough , RunnableLambda
def create_story_chain():
    
    ##tempelate for story generation
    story_prompt=ChatPromptTemplate.from_messages(
        [
            ("system", "You are a creative story-teller , write a short and engaging story based on the given theme , character and setting"),
            ("user","Theme:{theme}\n Main Character:{character} \n Setting:{setting}")
        ]
    )
    ##Tempelate for story analysis
    analysis_prompt=ChatPromptTemplate.from_messages(
        [
            ("system","You are a literary critic . Analyze the following story and provide insights"),
            ("user","{story}")
        ]
    )
    
    story_chain=(
        story_prompt | model | StrOutputParser()
    )
    
    def analyze_story(story_text):
        return {"story":story_text}
    
    
    analysis_chain=(
        story_chain
        |RunnableLambda(analyze_story)
        |analysis_prompt
        |model
        |StrOutputParser()
    )
    
    return analysis_chain

In [67]:
chain=create_story_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative story-teller , write a short and engaging story based on the given theme , character and setting'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme:{theme}\n Main Character:{character} \n Setting:{setting}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'te

In [68]:
result=chain.invoke(
    {
    "theme":"Sci-fi post apocalypse",
    "character" :"Akshat and Shivam , the two sole survivors , Akshat is intelligent and Shivam is strong",
    "setting":"A large green forest which is wide as the amazon and with a lot of challenges and accomplishments to be sustained by the survivors"
    
    }               
)
print("Story and Analysis:")
print(result)

Story and Analysis:
**Analysis of "The Last Hope"**

"The Last Hope" is a post-apocalyptic short story that explores the themes of survival, hope, and redemption in a world ravaged by environmental disaster. The narrative is set in a world where the once-blue skies are now a permanent gray, and the civilization has been reduced to rubble. The story revolves around the unlikely alliance between Akshat, a brilliant scientist, and Shivam, a skilled warrior, who are the last two survivors of humanity.

**Character Analysis**

Akshat and Shivam are well-crafted characters with distinct personalities. Akshat is a brilliant scientist who brings intellectualism and analytical thinking to their partnership. Shivam, on the other hand, is a skilled warrior who provides brawn and physical strength to their alliance. Their contrasting skills and personalities create a dynamic duo that complements each other perfectly, allowing them to tackle the challenges of the post-apocalyptic world.

**Symbolis

In [ ]:
##stringoutparser() is used to print the result directly , without giving the ai message wala placeholder